# Lite3 Hand Pose Detection & REST Publisher (Google Colab)

Welcome to **Part 2 of 2** in the Lite3 Control using Hand Pose Hands-on Session! Now that you have recorded your hand shapes into `gestures.json` using `register_pose.ipynb`, this notebook watches your webcam live, figures out which pose **class** you are making, and sends commands over the network to move the robot dog.

### Key Concepts

- **What is a "Class"?**
  A named category for a hand pose. You saved up to 9 classes in `gestures.json`:
  - **7 movement classes** (`forward`, `backward`, `left`, `right`, `turn_left`, `turn_right`, `stop`) send speed commands 10 times per second while held.
  - **2 posture classes** (`sit`, `stand`) send a single posture trigger when you switch into that hand pose.
- **What is an API?** *(Application Programming Interface)*
  Think of the robot's API like a **receptionist or waiter**: instead of you needing to program the robot's leg motors directly, the robot runs a small server listening for polite command messages and forwards them to the robot's control system.
- **What is an HTTP Request (`POST`) & JSON Payload?**
  Just like your browser uses `http://` to fetch web pages, programs send commands across a network using **HTTP requests**. An **HTTP `POST` request** delivers an order slip formatted in **JSON** (text inside curly braces `{...}`):
  ```json
  {"topic": "/cmd_vel", "type": "geometry_msgs/Twist", "payload": {"linear": {"x": 0.2, "y": 0, "z": 0}, ...}}
  ```
  - `topic`: Which channel on the robot to send the command to (`/cmd_vel` for movement speeds, `/simple_cmd` for `sit`/`stand`).
  - `type`: The template name the robot expects (`geometry_msgs/Twist` or `message_transformer/SimpleCMD`).
  - `payload`: The actual numbers (speeds or command code).
- **What are an IP Address, a Port, and `localhost`?**
  - **IP Address** (e.g., `192.168.1.100`): The robot's "street address" on your Wi-Fi network so your message goes to the right device.
  - **Port** (e.g., `:3000`): The specific "door number" on that device where the robot's API server is listening.
  - **`localhost`**: A special nickname that always means **"this exact computer I am typing on right now"** (use this if testing against a robot simulator or bridge running directly on your own laptop).
- **How Can Google Colab Reach a Robot on Local Wi-Fi? (`BROWSER_LOCAL_FETCH = True`)**
  Google Colab's Python code runs on a Google Cloud computer far away—not on the workshop Wi-Fi—so Python itself cannot reach `localhost` or `192.168.x.x`. However, **your web browser runs on your laptop** on the workshop Wi-Fi! With `BROWSER_LOCAL_FETCH = True` (already turned on), Colab recognizes your hand pose and passes the JSON order to **your browser**, which sends the HTTP `POST` straight over your local Wi-Fi to the robot.


## Block 1: Setup, Installation & API / Payload Config

Run each grey code box in order by clicking its **Play (`▶`)** button (or pressing **`Shift + Enter`**) and waiting for the green checkmark (`✓`).

### Beginner Step-by-Step Setup Checklist

1. **Upload `gestures.json` into Google Colab**:
   - Click the **folder icon (`Files`)** on the far-left sidebar of Colab.
   - Click the **Upload to session storage** button (page icon with an upward arrow).
   - Pick the `gestures.json` file you downloaded from `register_pose.ipynb` (usually in your computer's `Downloads` folder) and click **OK**.
2. **Join the Robot Wi-Fi & Check the Robot's Address**:
   - Connect your laptop to the workshop Wi-Fi and ask the instructor for the robot's **IP address** (e.g., `192.168.1.100`), or use `localhost` if the robot bridge runs on your laptop.
3. **Set `API_URL` in Step 1.2 Below**:
   - In Step 1.2, update **only the address between `http://` and `:3000`**—keep `http://`, `:3000`, and `/api/topics/publish` intact:
   ```python
   BROWSER_LOCAL_FETCH = True
   API_URL = "http://192.168.1.100:3000/api/topics/publish"  # robot on local Wi-Fi (replace with your robot's IP)
   # or:
   API_URL = "http://localhost:3000/api/topics/publish"      # bridge running on your own laptop
   ```
4. **One-Time Browser Permission for a `http://192.168.x.x` Address** *(not needed for `localhost`)*:
   - Because Colab loads over `https://`, browsers block requests to plain `http://192.168.x.x` LAN addresses by default:
     1. Click the **tune / lock icon** at the left of the browser URL bar (next to `colab.research.google.com`).
     2. Click **Site settings**.
     3. Scroll to **Insecure content** and change it to **Allow**.
     4. Go back to the Colab tab and click **Reload** when asked (if Chrome also asks about **Local network access**, click **Allow**).
5. **Optional Advanced Mode (Public HTTPS Tunnel, `BROWSER_LOCAL_FETCH = False`)**:
   - Only needed if you want Colab's remote Python VM to send the request directly through an HTTPS tunnel (`ngrok http 3000` or `cloudflared tunnel --url http://localhost:3000`) to `https://<your-tunnel-domain>/api/topics/publish`.

### Understanding the Robot Settings in Step 1.2
- **`TOLERANCE` (`0.15`)**: How closely your live hand must match a saved pose. Lower (`0.10`) = stricter; higher (`0.20`) = easier to trigger.
- **`CLASS_PAYLOADS` (`/cmd_vel` Speeds)**:
  | Parameter | Controls | Positive (`+`) | Negative (`-`) |
  |---|---|---|---|
  | `lx` | Forward / backward speed | Walk `forward` (`0.2`) | Walk `backward` (`-0.2`) |
  | `ly` | Sideways step speed | Step `left` (`0.1`) | Step `right` (`-0.1`) |
  | `az` | Turning rotation speed | Turn `left` (`0.3`) | Turn `right` (`-0.3`) |
- **`ZERO_TWIST` Safety Brake**: When your hand leaves the frame or shows `Unknown`, the notebook sends `0.0` for all speeds so the robot stops safely instead of continuing its last motion.
- **`SIMPLE_CMD_PAYLOADS`**: `stand` sends command code `0x21010202` and `sit` sends `0x21010203` to `/simple_cmd` once each time you switch into that pose.

### Step 1.1: Install Dependencies & Download Hand Landmarker Model


In [ ]:
!pip install -q mediapipe opencv-python-headless requests numpy

import urllib.request
from pathlib import Path

MODEL_PATH = Path("hand_landmarker.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)
if not MODEL_PATH.exists():
    print("Downloading hand_landmarker.task...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Model ready:", MODEL_PATH.resolve())

### Step 1.2: Configure `API_URL`, Topic Payloads & Tolerance

Edit **`API_URL`** on line 2 of the code cell below to point to your robot (replace `localhost` with the robot's IP address on Wi-Fi, e.g. `"http://192.168.1.100:3000/api/topics/publish"`), then click **Play (`▶`)**.


In [ ]:
# === API Endpoint & Twist Payload Configuration ===
API_URL = "http://localhost:3000/api/topics/publish"  # <-- Change 'localhost' to the robot's Wi-Fi IP if needed (e.g. 192.168.1.100)
BROWSER_LOCAL_FETCH = True  # True: browser sends HTTP POST over your local Wi-Fi (recommended in Colab); False: Python requests.post
TOPIC = "/cmd_vel"          # Robot channel for continuous walking/turning velocities
MSG_TYPE = "geometry_msgs/Twist"
SIMPLE_CMD_TOPIC = "/simple_cmd"  # Robot channel for one-shot posture commands (sit/stand)
SIMPLE_CMD_TYPE = "message_transformer/SimpleCMD"
STAND_CMD_CODE = 0x21010202  # 553714178 (stand up)
SIT_CMD_CODE = 0x21010203    # 553714179 (sit / lie down)

PUBLISH_HZ = 10
PUBLISH_INTERVAL = 1.0 / PUBLISH_HZ  # 0.1s between velocity commands (10 times per second)
HTTP_TIMEOUT = 0.1

GESTURES_PATH = "gestures.json"  # File uploaded to Colab Files panel from register_pose.ipynb
TOLERANCE = 0.15                 # Max difference score allowed to match a pose (lower = stricter)


def make_twist(lx: float = 0.0, ly: float = 0.0, lz: float = 0.0, az: float = 0.0) -> dict:
    return {
        "linear": {"x": lx, "y": ly, "z": lz},
        "angular": {"x": 0.0, "y": 0.0, "z": az},
    }


def make_simple_cmd(cmd_code: int, cmd_value: int = 0, cmd_type: int = 0) -> dict:
    return {"cmd_code": cmd_code, "cmd_value": cmd_value, "type": cmd_type}


CLASS_PAYLOADS: dict[str, dict] = {
    "forward": make_twist(lx=0.2),
    "backward": make_twist(lx=-0.2),
    "left": make_twist(ly=0.1),
    "right": make_twist(ly=-0.1),
    "turn_left": make_twist(az=0.3),
    "turn_right": make_twist(az=-0.3),
    "stop": make_twist(),
}
SIMPLE_CMD_PAYLOADS: dict[str, dict] = {
    "sit": make_simple_cmd(SIT_CMD_CODE),
    "stand": make_simple_cmd(STAND_CMD_CODE),
}
ZERO_TWIST = make_twist()

### Step 1.3: Landmark Normalization, Gesture Classifier & HTTP Publisher

Click **Play (`▶`)** to load the gesture-matching math and HTTP request builder, and check the output line printed below the cell:
- **`Loaded gestures: ['forward', ...]`**: Your `gestures.json` file was found and loaded!
- **`Loaded gestures: []`**: `gestures.json` is missing—upload it using the 📁 **Files** sidebar before running Block 2.


In [ ]:
import base64
import json
import time
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import requests
from mediapipe.tasks.python import vision

try:
    BaseOptions = mp.tasks.BaseOptions
except AttributeError:
    from mediapipe.tasks.python import core
    BaseOptions = core.base_options.BaseOptions

HAND_CONNECTIONS = (
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
)


def draw_hand_skeleton(frame: np.ndarray, hand_landmarks) -> None:
    if hasattr(vision, "drawing_utils") and hasattr(vision, "drawing_styles") and hasattr(vision, "HandLandmarksConnections"):
        vision.drawing_utils.draw_landmarks(
            frame,
            hand_landmarks,
            vision.HandLandmarksConnections.HAND_CONNECTIONS,
            vision.drawing_styles.get_default_hand_landmarks_style(),
            vision.drawing_styles.get_default_hand_connections_style(),
        )
        return
    h, w = frame.shape[:2]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 220, 0), 2, cv2.LINE_AA)
    for x, y in pts:
        cv2.circle(frame, (x, y), 4, (0, 80, 255), -1, cv2.LINE_AA)


def normalize_landmarks(hand_landmarks) -> np.ndarray:
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks], dtype=np.float32)
    centered = coords - coords[0]
    scale = float(np.max(np.linalg.norm(centered, axis=1)))
    return centered / scale if scale > 1e-6 else centered


def load_gestures(path: str | Path = GESTURES_PATH) -> dict[str, np.ndarray]:
    p = Path(path)
    if not p.exists():
        return {}
    raw = json.loads(p.read_text(encoding="utf-8"))
    return {name: np.asarray(coords, dtype=np.float32) for name, coords in raw.items()}


def classify_gesture(
    live_coords: np.ndarray,
    gestures: dict[str, np.ndarray],
    tolerance: float = TOLERANCE,
) -> tuple[str, float | None]:
    if not gestures:
        return "No gestures saved", None
    best_name, best_dist = None, float("inf")
    for name, ref_coords in gestures.items():
        if ref_coords.shape != live_coords.shape:
            continue
        dist = float(np.mean(np.linalg.norm(live_coords - ref_coords, axis=1)))
        if dist < best_dist:
            best_dist, best_name = dist, name
    if best_name is None:
        return "No gestures saved", None
    return (best_name, best_dist) if best_dist <= tolerance else ("Unknown", best_dist)


def build_publish_body(class_name: str | None) -> tuple[dict, str]:
    if class_name in SIMPLE_CMD_PAYLOADS:
        topic, msg_type = SIMPLE_CMD_TOPIC, SIMPLE_CMD_TYPE
        payload, label = SIMPLE_CMD_PAYLOADS[class_name], class_name
    else:
        topic, msg_type = TOPIC, MSG_TYPE
        is_known = class_name in CLASS_PAYLOADS
        payload = CLASS_PAYLOADS[class_name] if is_known else ZERO_TWIST
        label = class_name if is_known else "zero"
    return {"topic": topic, "type": msg_type, "payload": payload}, label


def publish_cmd_vel(class_name: str | None) -> str:
    body, label = build_publish_body(class_name)
    try:
        resp = requests.post(API_URL, json=body, timeout=HTTP_TIMEOUT)
        return f"HTTP {resp.status_code}: {label}"
    except requests.RequestException as exc:
        return f"HTTP err ({label}): {exc}"


def decode_data_url(data_url: str) -> np.ndarray | None:
    if not data_url or "," not in data_url:
        return None
    encoded = data_url.split(",", 1)[1]
    if not encoded:
        return None
    buf = np.frombuffer(base64.b64decode(encoded), dtype=np.uint8)
    if buf.size == 0:
        return None
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)


def encode_data_url(frame: np.ndarray) -> str:
    ok, buf = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
    if not ok:
        raise RuntimeError("Failed to encode frame")
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")


def create_image_landmarker() -> vision.HandLandmarker:
    options = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=1,
    )
    return vision.HandLandmarker.create_from_options(options)


print("Loaded gestures:", list(load_gestures().keys()))

## Block 2: Running (Live Detection & REST API Publish)

Click **Play (`▶`)** on the cell below and allow webcam access to start controlling the robot dog live!

### What Happens While Running
1. **Webcam & Hand Tracking**: Your browser captures mirrored `640x480` frames and sends them to MediaPipe, which locates 21 hand dots and draws your skeleton.
2. **Pose Matching**: Your live hand pose is compared against the classes you saved in `gestures.json`.
3. **Sending HTTP `POST` Commands (`BROWSER_LOCAL_FETCH = True`)**:
   - Python builds the JSON command payload and passes it back to **your browser**, which sends the HTTP `POST` request to `API_URL` directly over your local Wi-Fi or `localhost`.
   - **Movement classes** (`forward`, `backward`, `left`, `right`, `turn_left`, `turn_right`, `stop`) publish to `/cmd_vel` **10 times per second** (and send safe all-zero `ZERO_TWIST` speeds whenever no hand is seen or the pose is `Unknown`).
   - **Posture classes** (`sit`, `stand`) publish **once** to `/simple_cmd` each time your hand switches into `sit` or `stand`.

### How to Read the Live Status Line
Look above or at the bottom of the video feed—for example: **`Gesture: forward (err=0.062) | Browser POST -> forward (OK)`**
- **`Gesture: forward (err=0.062)`**: Matched the `forward` class with distance error `0.062` (under your `TOLERANCE` of `0.15`).
- **`Browser POST -> forward (OK)`** (or **`Browser POST (no-cors) -> ...`**): Your browser delivered the HTTP `POST` request over the network to the robot bridge! *(Note: `no-cors` still delivers the JSON command when the robot bridge does not send CORS browser headers.)*
- Click **STOP CAMERA** when finished to turn off your webcam and return the robot to idle.

### Beginner Troubleshooting Guide
- **`Loaded gestures: []` or `Gesture: No gestures saved`**: Colab cannot find `gestures.json` in the current session. Open the 📁 **Files** sidebar on the left, upload `gestures.json`, and re-run Step 1.3 + Block 2.
- **`Gesture: Unknown`**: Your current hand shape didn't closely match any pose in `gestures.json` within `TOLERANCE`. Hold your pose steadier inside the frame, try a slightly higher `TOLERANCE` (like `0.20` in Step 1.2), or re-record that class cleanly in `register_pose.ipynb`.
- **`Browser POST err ... Failed to fetch`**: Your browser could not reach `API_URL`. Check that:
  1. The robot dog (and its REST API bridge on port `3000`) is powered on and your laptop is on the **same Wi-Fi**.
  2. `API_URL` in Step 1.2 has the exact right IP address and port (e.g., `http://192.168.1.100:3000/api/topics/publish`), and you re-ran Step 1.2 after editing it.
  3. If `API_URL` uses `http://192.168.x.x`, you allowed **Insecure content** in Chrome **Site settings** for `colab.research.google.com` and reloaded the page.
- **Camera not starting / already in use**: Close `register_pose.ipynb`'s camera (**STOP**) or other video apps, allow webcam permission in the browser URL bar, and re-run Block 2.


In [ ]:
from google.colab import output
from IPython.display import HTML, JSON, display

_det_landmarker = create_image_landmarker()
_det_gestures = load_gestures(GESTURES_PATH)
_last_pub_class = "__init__"
_last_pub_time = 0.0
_last_api_status = "API: idle"


def _detect_frame_cb(data_url: str, client_api_status: str = ""):
    global _det_gestures, _last_pub_class, _last_pub_time, _last_api_status
    _det_gestures = load_gestures(GESTURES_PATH)
    if BROWSER_LOCAL_FETCH and client_api_status:
        _last_api_status = client_api_status

    frame = decode_data_url(data_url)
    if frame is None:
        return JSON({"image": "", "status": f"Waiting for valid frame... | {_last_api_status}", "publish": None})

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = _det_landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))

    detected_label = None
    banner = "Hand: none"
    color = (180, 180, 180)

    if res.hand_landmarks:
        hand_lms = res.hand_landmarks[0]
        draw_hand_skeleton(frame, hand_lms)
        coords = normalize_landmarks(hand_lms)
        label, dist = classify_gesture(coords, _det_gestures, TOLERANCE)
        detected_label = label if (label in CLASS_PAYLOADS or label in SIMPLE_CMD_PAYLOADS) else None
        banner = f"Gesture: {label}" + (f" (err={dist:.3f})" if dist is not None else "")
        color = (0, 220, 0) if detected_label is not None else (0, 180, 255)

    now = time.monotonic()
    is_simple = detected_label in SIMPLE_CMD_PAYLOADS
    should_pub = (
        (detected_label != _last_pub_class)
        if is_simple
        else (detected_label != _last_pub_class or now - _last_pub_time >= PUBLISH_INTERVAL)
    )
    publish_req = None
    if should_pub:
        if BROWSER_LOCAL_FETCH:
            body, label = build_publish_body(detected_label)
            publish_req = {"url": API_URL, "body": body, "label": label}
            if not client_api_status:
                _last_api_status = f"Browser POST -> {label}"
        else:
            _last_api_status = publish_cmd_vel(detected_label)
        _last_pub_class = detected_label
        _last_pub_time = now

    cv2.putText(frame, banner, (16, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    cv2.putText(
        frame,
        _last_api_status[:55],
        (16, frame.shape[0] - 14),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.52,
        (220, 220, 220),
        1,
        cv2.LINE_AA,
    )
    return JSON({
        "image": encode_data_url(frame),
        "status": f"{banner} | {_last_api_status}",
        "publish": publish_req,
    })


output.register_callback("detect_frame_cb", _detect_frame_cb)

display(HTML("""
<div style="font-family:sans-serif; max-width:720px;">
  <div style="margin-bottom:8px;">
    <button onclick="stopDetCam()" style="padding:6px 14px; background:#c62828; color:#fff; border:none; cursor:pointer; font-weight:bold;">STOP CAMERA</button>
    <span id="det-status" style="margin-left:10px; font-weight:bold;">Starting camera...</span>
  </div>
  <video id="det-vid" autoplay playsinline style="display:none;"></video>
  <img id="det-img" style="width:100%; max-width:640px; border:2px solid #444;" />
</div>
<script>
(() => {
  if (window._robodogStopCam) window._robodogStopCam();
  let running = true;
  let stream = null;
  let lastApiStatus = "";
  const video = document.getElementById("det-vid");
  const img = document.getElementById("det-img");
  const statusEl = document.getElementById("det-status");
  const canvas = document.createElement("canvas");
  canvas.width = 640;
  canvas.height = 480;
  const ctx = canvas.getContext("2d");

  const stopFn = () => {
    running = false;
    if (stream) stream.getTracks().forEach(t => t.stop());
    if (window._robodogStopCam === stopFn) window._robodogStopCam = null;
    statusEl.textContent = "Camera stopped.";
  };
  window.stopDetCam = stopFn;
  window._robodogStopCam = stopFn;

  async function sendLocalApi(pub) {
    const payload = JSON.stringify(pub.body);
    try {
      const resp = await fetch(pub.url, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: payload,
      });
      lastApiStatus = resp.ok
        ? "Browser POST -> " + pub.label + " (OK)"
        : "Browser HTTP " + resp.status + ": " + pub.label;
    } catch (err) {
      try {
        await fetch(pub.url, {
          method: "POST",
          mode: "no-cors",
          headers: { "Content-Type": "text/plain" },
          body: payload,
        });
        lastApiStatus = "Browser POST (no-cors) -> " + pub.label;
      } catch (err2) {
        lastApiStatus = "Browser POST err (" + pub.label + "): " + err2;
      }
    }
  }

  async function loop() {
    try {
      stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
      video.srcObject = stream;
      await video.play();
      while (running) {
        if (video.readyState < 2 || video.videoWidth === 0) {
          await new Promise(r => setTimeout(r, 30));
          continue;
        }
        ctx.save();
        ctx.translate(canvas.width, 0);
        ctx.scale(-1, 1);
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        ctx.restore();
        const dataUrl = canvas.toDataURL("image/jpeg", 0.8);
        const res = await google.colab.kernel.invokeFunction("detect_frame_cb", [dataUrl, lastApiStatus], {});
        let out = res?.data?.["application/json"];
        if (typeof out === "string") out = JSON.parse(out);
        if (out) {
          if (out.image) img.src = out.image;
          statusEl.textContent = out.status;
          if (out.publish) sendLocalApi(out.publish);
        }
        await new Promise(r => setTimeout(r, 30));
      }
    } catch (err) {
      statusEl.textContent = "Camera error: " + err;
    }
  }
  loop();
})();
</script>
"""))